In [1]:
import pandas as pd
import numpy as np
calendar = pd.read_csv("calendar_processed.csv",parse_dates=['date'])
sales_train = pd.read_csv('sales_train.csv',parse_dates=['date'])
sales_test = pd.read_csv('sales_test.csv',parse_dates=['date'])
inventory = pd.read_csv('inventory.csv')

/Users/lingyizhao/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
calendar

,date,holiday_name,shops_closed,winter_school_holidays,school_holidays,warehouse,year,month,day,day_of_week,...,all_saints'_day_holiday,ascension_day,corpus_christi,german_unity_day,reformation_day,epiphany,assumption_of_the_virgin_mary,peace_festival_in_augsburg,day_before_holiday,day_after_holiday
0,2016-01-01,Not,1,0,0,Brno_1,2016,1,1,4,...,0,0,0,0,0,0,0,0,1.0,-1.0
1,2016-01-02,Not,0,0,0,Brno_1,2016,1,2,5,...,0,0,0,0,0,0,0,0,1.0,1.0
2,2016-01-03,Not,0,0,0,Brno_1,2016,1,3,6,...,0,0,0,0,0,0,0,0,1.0,1.0
3,2016-01-04,Not,0,0,0,Brno_1,2016,1,4,0,...,0,0,0,0,0,0,0,0,1.0,1.0
4,2016-01-05,Not,0,0,0,Brno_1,2016,1,5,1,...,0,0,0,0,0,0,0,0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23011,2024-12-27,Not,0,0,0,Prague_3,2024,12,27,4,...,0,0,0,0,0,0,0,0,1.0,1.0
23012,2024-12-28,Not,0,0,0,Prague_3,2024,12,28,5,...,0,0,0,0,0,0,0,0,1.0,1.0
23013,2024-12-29,Not,0,0,0,Prague_3,2024,12,29,6,...,0,0,0,0,0,0,0,0,1.0,1.0
23014,2024-12-30,Not,0,0,0,Prague_3,2024,12,30,0,...,0,0,0,0,0,0,0,0,1.0,1.0


In [3]:
calendar = pd.read_csv("calendar_processed.csv",parse_dates=['date'])

In [4]:
calendar.dtypes

date                                                           datetime64[ns]
holiday_name                                                           object
shops_closed                                                            int64
winter_school_holidays                                                  int64
school_holidays                                                         int64
warehouse                                                              object
year                                                                    int64
month                                                                   int64
day                                                                     int64
day_of_week                                                             int64
new_years_day                                                           int64
international_womens_day                                                int64
good_friday                                                     

In [5]:
calendar.columns

Index(['date', 'holiday_name', 'shops_closed', 'winter_school_holidays',
       'school_holidays', 'warehouse', 'year', 'month', 'day', 'day_of_week',
       'new_years_day', 'international_womens_day', 'good_friday',
       'holy_saturday', 'easter_day', 'easter_monday', 'labour_day',
       'mother's_day', 'cyrila_a_metodej', 'jan_hus', 'den_ceske_statnosti',
       'den_vzniku_samostatneho_ceskoslovenskeho_statu',
       'den_boje_za_svobodu_a_demokracii', 'christmas_eve',
       '1st_christmas_day', '2nd_christmas_day', 'den_osvobozeni',
       'memorial_day_of_the_republic',
       'memorial_day_for_the_victims_of_the_communist_dictatorships',
       'memorial_day_for_the_victims_of_the_holocaust', 'whit_sunday',
       'whit_monday', 'national_defense_day', 'day_of_national_unity',
       'independent_hungary_day', 'state_foundation_day',
       'memorial_day_for_the_martyrs_of_arad',
       'memorial_day_of_the_1956_revolution', 'all_saints_day',
       'hungary_national_day_hol

In [8]:
calendar = calendar[['date','holiday_name','shops_closed','winter_school_holidays',
       'school_holidays', 'warehouse','year','month', 'day','day_of_week','day_before_holiday', 'day_after_holiday']]
calendar

,date,holiday_name,shops_closed,winter_school_holidays,school_holidays,warehouse,year,month,day,day_of_week,day_before_holiday,day_after_holiday
0,2016-01-01,Not,1,0,0,Brno_1,2016,1,1,4,1.0,-1.0
1,2016-01-02,Not,0,0,0,Brno_1,2016,1,2,5,1.0,1.0
2,2016-01-03,Not,0,0,0,Brno_1,2016,1,3,6,1.0,1.0
3,2016-01-04,Not,0,0,0,Brno_1,2016,1,4,0,1.0,1.0
4,2016-01-05,Not,0,0,0,Brno_1,2016,1,5,1,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
23011,2024-12-27,Not,0,0,0,Prague_3,2024,12,27,4,1.0,1.0
23012,2024-12-28,Not,0,0,0,Prague_3,2024,12,28,5,1.0,1.0
23013,2024-12-29,Not,0,0,0,Prague_3,2024,12,29,6,1.0,1.0
23014,2024-12-30,Not,0,0,0,Prague_3,2024,12,30,0,1.0,1.0


## 合并数据

In [9]:
#合并 sales train, sales test与inventory
inventory_subset = inventory[['unique_id', 'product_unique_id', 'warehouse']]

# Merge with sales_train and sales_test
sales_train = pd.merge(sales_train, inventory_subset, how='left', on=['unique_id', 'warehouse'])
sales_test = pd.merge(sales_test, inventory_subset, how='left', on=['unique_id', 'warehouse'])


In [10]:
#处理sales na step1:什么都有只没有sales
sales_train.loc[sales_train['total_orders'].notna() & sales_train['sell_price_main'].notna(), 'sales'] = sales_train['sales'].fillna(0)

## 小智helpme

In [11]:
#sales_test和sales_trains处理,drop availability,合并discounts,和calendar merge
def process_sales_data(sales_data, calendar):
    discount_cols = [f"type_{i}_discount" for i in range(7)]
    sales_data = pd.merge(sales_data, calendar, how='left', on=['date', 'warehouse'])
    sales_data["max_discount"] = sales_data[discount_cols].max(axis=1).fillna(1)
    
    # Drop discount columns
    drop_cols = discount_cols + ["availability"] if "availability" in sales_data.columns else discount_cols
    return sales_data.drop(columns=drop_cols, errors='ignore')

sales_train = process_sales_data(sales_train, calendar)
sales_test = process_sales_data(sales_test, calendar)



In [5]:
# One-hot encode the 'warehouse' column

encoded_warehouses_test = pd.get_dummies(sales_test['warehouse'], prefix='warehouse')
sales_test_encoded = pd.concat([sales_test.drop('warehouse', axis=1), encoded_warehouses_test], axis=1)
sales_test_encoded =sales_test_encoded.drop('holiday_name', axis=1)
sales_test_encoded

,unique_id,date,total_orders,sell_price_main,product_unique_id,shops_closed,winter_school_holidays,school_holidays,year,month,...,day_before_holiday,day_after_holiday,max_discount,warehouse_Brno_1,warehouse_Budapest_1,warehouse_Frankfurt_1,warehouse_Munich_1,warehouse_Prague_1,warehouse_Prague_2,warehouse_Prague_3
0,1226,2024-06-03,8679,13.13,627,0,0,0,2024,6,...,1.0,1.0,0.0,1,0,0,0,0,0,0
1,5409,2024-06-03,5705,58.26,2661,0,0,0,2024,6,...,1.0,1.0,0.0,0,0,0,0,0,1,0
2,1268,2024-06-03,5192,19.68,648,0,0,0,2024,6,...,1.0,1.0,0.0,0,0,0,0,0,0,1
3,1531,2024-06-03,5209,29.70,778,0,0,0,2024,6,...,1.0,1.0,0.0,0,0,0,0,0,0,1
4,655,2024-06-03,5705,21.40,332,0,0,0,2024,6,...,1.0,1.0,0.0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47016,368,2024-06-16,6113,1094.42,185,0,0,0,2024,6,...,1.0,1.0,0.0,0,1,0,0,0,0,0
47017,1718,2024-06-16,6113,555.03,866,0,0,0,2024,6,...,1.0,1.0,0.0,0,1,0,0,0,0,0
47018,3831,2024-06-16,8453,148.22,1883,0,0,0,2024,6,...,1.0,1.0,0.0,1,0,0,0,0,0,0
47019,2629,2024-06-16,10264,49.43,1311,0,0,0,2024,6,...,1.0,1.0,0.0,0,0,0,0,1,0,0


In [12]:
sales_test.columns

Index(['unique_id', 'date', 'warehouse', 'total_orders', 'sell_price_main',
       'product_unique_id', 'holiday_name', 'shops_closed',
       'winter_school_holidays', 'school_holidays', 'year', 'month', 'day',
       'day_of_week', 'day_before_holiday', 'day_after_holiday',
       'max_discount'],
      dtype='object')

In [13]:
sales_test=sales_test.drop('date', axis=1)

In [14]:
sales_test.to_csv('not_encoded_sales_test.csv', index=False)

In [15]:
sales_test

,unique_id,warehouse,total_orders,sell_price_main,product_unique_id,holiday_name,shops_closed,winter_school_holidays,school_holidays,year,month,day,day_of_week,day_before_holiday,day_after_holiday,max_discount
0,1226,Brno_1,8679,13.13,627,Not,0,0,0,2024,6,3,0,1.0,1.0,0.0
1,5409,Prague_2,5705,58.26,2661,Not,0,0,0,2024,6,3,0,1.0,1.0,0.0
2,1268,Prague_3,5192,19.68,648,Not,0,0,0,2024,6,3,0,1.0,1.0,0.0
3,1531,Prague_3,5209,29.70,778,Not,0,0,0,2024,6,3,0,1.0,1.0,0.0
4,655,Prague_2,5705,21.40,332,Not,0,0,0,2024,6,3,0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47016,368,Budapest_1,6113,1094.42,185,Not,0,0,0,2024,6,16,6,1.0,1.0,0.0
47017,1718,Budapest_1,6113,555.03,866,Not,0,0,0,2024,6,16,6,1.0,1.0,0.0
47018,3831,Brno_1,8453,148.22,1883,Not,0,0,0,2024,6,16,6,1.0,1.0,0.0
47019,2629,Prague_1,10264,49.43,1311,Not,0,0,0,2024,6,16,6,1.0,1.0,0.0


In [16]:
sales_train.columns

Index(['unique_id', 'date', 'warehouse', 'total_orders', 'sales',
       'sell_price_main', 'product_unique_id', 'holiday_name', 'shops_closed',
       'winter_school_holidays', 'school_holidays', 'year', 'month', 'day',
       'day_of_week', 'day_before_holiday', 'day_after_holiday',
       'max_discount'],
      dtype='object')

In [17]:
sales_train

,unique_id,date,warehouse,total_orders,sales,sell_price_main,product_unique_id,holiday_name,shops_closed,winter_school_holidays,school_holidays,year,month,day,day_of_week,day_before_holiday,day_after_holiday,max_discount
0,4845,2024-03-10,Budapest_1,6436.0,16.34,646.26,2375,Not,0,0,0,2024,3,10,6,1.0,1.0,0.15312
1,4845,2021-05-25,Budapest_1,4663.0,12.63,455.96,2375,Not,0,0,0,2021,5,25,1,1.0,1.0,0.15025
2,4845,2021-12-20,Budapest_1,6507.0,34.55,455.96,2375,Not,0,0,0,2021,12,20,0,1.0,1.0,0.15025
3,4845,2023-04-29,Budapest_1,5463.0,34.52,646.26,2375,Not,0,0,0,2023,4,29,5,1.0,1.0,0.20024
4,4845,2022-04-01,Budapest_1,5997.0,35.92,486.41,2375,good_friday,0,0,0,2022,4,1,4,1.0,1.0,0.15649
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4007414,4941,2023-06-21,Prague_1,9988.0,26.56,34.06,2422,Not,0,0,0,2023,6,21,2,1.0,1.0,0.00000
4007415,4941,2023-06-24,Prague_1,8518.0,27.42,34.06,2422,Not,0,0,0,2023,6,24,5,1.0,1.0,0.00000
4007416,4941,2023-06-23,Prague_1,10424.0,33.39,34.06,2422,Not,0,0,0,2023,6,23,4,1.0,1.0,0.00000
4007417,4941,2023-06-22,Prague_1,10342.0,22.88,34.06,2422,Not,0,0,0,2023,6,22,3,1.0,1.0,0.00000


## 继续处理sales na

In [18]:
#处理sales na step2:shop_closed == 1
# Fill sales with 0 if shops_closed == 1 and sales is NaN
sales_train.loc[(sales_train['shops_closed'] == 1) & (sales_train['sales'].isna()), 'sales'] = 0

# group sales_train by date, Fill missing price using the previous day's price within the same warehouse and unique_id
sales_train['sell_price_main'] = sales_train.groupby(['warehouse', 'unique_id'])['sell_price_main'].ffill()

In [19]:
missing_dates = sales_train[sales_train['sales'].isna()]['date'].unique()
missing_dates

<DatetimeArray>
['2021-05-21 00:00:00', '2021-05-22 00:00:00', '2021-12-10 00:00:00',
 '2021-12-09 00:00:00', '2021-06-27 00:00:00', '2021-06-26 00:00:00',
 '2021-05-29 00:00:00', '2021-05-31 00:00:00', '2021-05-30 00:00:00',
 '2021-06-20 00:00:00', '2021-06-12 00:00:00', '2021-07-04 00:00:00',
 '2021-06-19 00:00:00', '2021-06-13 00:00:00', '2021-07-11 00:00:00']
Length: 15, dtype: datetime64[ns]

In [20]:
#处理sales na step3.5:shop_closed == 0
# Fill sales with 0 if shops_closed == 1 and sales is NaN

sales_train.loc[(sales_train['date'].isin(missing_dates)) & (sales_train['sales'].isna()), 'sales'] = 0

In [14]:
#处理sales na step4:疫情
# Get all relevant dates from missing_combinations
#missing_dates = # missing dates in sales_train

# Define the conditions for sales_train where sales should be filled
#condition_munich = (sales_train['warehouse'] == 'Munich_1') & (sales_train['date'].isin(missing_dates))
#condition_frankfurt = (sales_train['warehouse'] == 'Frankfurt_1') & (sales_train['date'].isin(missing_dates))

# Combine conditions
#condition = condition_munich | condition_frankfurt

# Compute the average sales for the same product_unique_id and date across other warehouses
#avg_sales = sales_train.groupby(['product_unique_id', 'date'])['sales'].transform(lambda x: x.mean())

# Fill sales where the condition is met and sales is NaN
#sales_train.loc[condition & sales_train['sales'].isna(), 'sales'] = avg_sales

In [21]:
#处理sales na step4:其他
# Step 2: For the rest of missing_dates in missing_combinations:
# - Set sales to 0
# - Fill total_orders using same day's value within the same warehouse, then use previous day's value if still missing
# - Fill price using previous day's price within the same warehouse and unique_id

# Identify remaining missing sales
remaining_condition = sales_train['date'].isin(missing_dates) & sales_train['sales'].isna()

# Fill sales with 0
sales_train.loc[remaining_condition, 'sales'] = 0

# Fill total_orders using same day's value within the same warehouse
sales_train['total_orders'] = sales_train.groupby(['date', 'warehouse'])['total_orders'].transform(lambda x: x.fillna(method='ffill'))

# Fill remaining missing total_orders using previous day's value within the same warehouse
sales_train['total_orders'] = sales_train.groupby('warehouse')['total_orders'].ffill()

# Fill price using previous day's price within the same warehouse and unique_id
sales_train['sell_price_main'] = sales_train.groupby(['warehouse', 'unique_id'])['sell_price_main'].ffill()

/var/folders/m9/_gpbs0bn61q7077ldd342jc00000gn/T/ipykernel_97918/1885470117.py:14: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  sales_train['total_orders'] = sales_train.groupby(['date', 'warehouse'])['total_orders'].transform(lambda x: x.fillna(method='ffill'))


In [16]:
encoded_warehouses = pd.get_dummies(sales_train['warehouse'], prefix='warehouse')
sales_train_encoded = pd.concat([sales_train.drop('warehouse', axis=1), encoded_warehouses], axis=1)
#sales_train_encoded.columns


sales_train_encoded=sales_train_encoded.drop('holiday_name',axis=1)
sales_train_encoded


,unique_id,date,total_orders,sales,sell_price_main,product_unique_id,shops_closed,winter_school_holidays,school_holidays,year,...,day_before_holiday,day_after_holiday,max_discount,warehouse_Brno_1,warehouse_Budapest_1,warehouse_Frankfurt_1,warehouse_Munich_1,warehouse_Prague_1,warehouse_Prague_2,warehouse_Prague_3
0,4845,2024-03-10,6436.0,16.34,646.26,2375,0,0,0,2024,...,1.0,1.0,0.15312,0,1,0,0,0,0,0
1,4845,2021-05-25,4663.0,12.63,455.96,2375,0,0,0,2021,...,1.0,1.0,0.15025,0,1,0,0,0,0,0
2,4845,2021-12-20,6507.0,34.55,455.96,2375,0,0,0,2021,...,1.0,1.0,0.15025,0,1,0,0,0,0,0
3,4845,2023-04-29,5463.0,34.52,646.26,2375,0,0,0,2023,...,1.0,1.0,0.20024,0,1,0,0,0,0,0
4,4845,2022-04-01,5997.0,35.92,486.41,2375,0,0,0,2022,...,1.0,1.0,0.15649,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4007414,4941,2023-06-21,9988.0,26.56,34.06,2422,0,0,0,2023,...,1.0,1.0,0.00000,0,0,0,0,1,0,0
4007415,4941,2023-06-24,8518.0,27.42,34.06,2422,0,0,0,2023,...,1.0,1.0,0.00000,0,0,0,0,1,0,0
4007416,4941,2023-06-23,10424.0,33.39,34.06,2422,0,0,0,2023,...,1.0,1.0,0.00000,0,0,0,0,1,0,0
4007417,4941,2023-06-22,10342.0,22.88,34.06,2422,0,0,0,2023,...,1.0,1.0,0.00000,0,0,0,0,1,0,0


In [24]:
#print(sales_train[sales_train['sales'].isna()])
#print(sales_train_encoded[sales_train_encoded.isna()])
# Print all rows where at least one column has a NaN value
#print(sales_train_encoded[sales_train_encoded.isna().any(axis=1)])
print(sales_train[sales_train.isna().any(axis=1)])


Empty DataFrame
Columns: [unique_id, date, warehouse, total_orders, sales, sell_price_main, product_unique_id, holiday_name, shops_closed, winter_school_holidays, school_holidays, year, month, day, day_of_week, day_before_holiday, day_after_holiday, max_discount]
Index: []


In [ ]:
#sales_train_encoded.to_csv('processed_sales_train.csv', index=False)

In [19]:
sales_train_encoded

,unique_id,date,total_orders,sales,sell_price_main,product_unique_id,shops_closed,winter_school_holidays,school_holidays,year,...,day_before_holiday,day_after_holiday,max_discount,warehouse_Brno_1,warehouse_Budapest_1,warehouse_Frankfurt_1,warehouse_Munich_1,warehouse_Prague_1,warehouse_Prague_2,warehouse_Prague_3
0,4845,2024-03-10,6436.0,16.34,646.26,2375,0,0,0,2024,...,1.0,1.0,0.15312,0,1,0,0,0,0,0
1,4845,2021-05-25,4663.0,12.63,455.96,2375,0,0,0,2021,...,1.0,1.0,0.15025,0,1,0,0,0,0,0
2,4845,2021-12-20,6507.0,34.55,455.96,2375,0,0,0,2021,...,1.0,1.0,0.15025,0,1,0,0,0,0,0
3,4845,2023-04-29,5463.0,34.52,646.26,2375,0,0,0,2023,...,1.0,1.0,0.20024,0,1,0,0,0,0,0
4,4845,2022-04-01,5997.0,35.92,486.41,2375,0,0,0,2022,...,1.0,1.0,0.15649,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4007414,4941,2023-06-21,9988.0,26.56,34.06,2422,0,0,0,2023,...,1.0,1.0,0.00000,0,0,0,0,1,0,0
4007415,4941,2023-06-24,8518.0,27.42,34.06,2422,0,0,0,2023,...,1.0,1.0,0.00000,0,0,0,0,1,0,0
4007416,4941,2023-06-23,10424.0,33.39,34.06,2422,0,0,0,2023,...,1.0,1.0,0.00000,0,0,0,0,1,0,0
4007417,4941,2023-06-22,10342.0,22.88,34.06,2422,0,0,0,2023,...,1.0,1.0,0.00000,0,0,0,0,1,0,0


In [25]:


# Identify columns in both DataFrames
columns_train = set(sales_train.columns)
columns_test = set(sales_test.columns)

# Find common columns
common_columns = columns_train.intersection(columns_test)

# Find columns that are only in sales_train
only_in_train = columns_train - columns_test

# Find columns that are only in sales_test
only_in_test = columns_test - columns_train

# Print the results
print("Common Columns:")
print(common_columns)

print("\nColumns Only in sales_train:")
print(only_in_train)

print("\nColumns Only in sales_test:")
print(only_in_test)


Common Columns:
{'day_before_holiday', 'total_orders', 'sell_price_main', 'winter_school_holidays', 'warehouse', 'school_holidays', 'year', 'day_after_holiday', 'max_discount', 'holiday_name', 'month', 'unique_id', 'product_unique_id', 'shops_closed', 'day', 'day_of_week'}

Columns Only in sales_train:
{'date', 'sales'}

Columns Only in sales_test:
set()


In [21]:
# # Step 1: Get all unique combinations of warehouse and date from the calendar
# #unique_combinations_calendar = calendar[['warehouse', 'date']].drop_duplicates()

# # Step 2: Process each warehouse separately
# #result_frames = []

# #for warehouse in unique_combinations_calendar['warehouse'].unique():
#     # Filter data for the current warehouse in calendar and sales_train
#     calendar_data = unique_combinations_calendar[unique_combinations_calendar['warehouse'] == warehouse]
#     warehouse_sales_data = sales_train[sales_train['warehouse'] == warehouse]

#     # Determine the date range for this warehouse from sales_train
#     start_date = warehouse_sales_data['date'].min()
#     end_date = warehouse_sales_data['date'].max()

#     # Filter calendar data to match the date range of sales_train
#     warehouse_dates = calendar_data[(calendar_data['date'] >= start_date) & (calendar_data['date'] <= end_date)]['date'].unique()

#     # Generate unique combinations of unique_id and date for this warehouse
#     full_combinations = pd.MultiIndex.from_product(
#         [warehouse_sales_data['unique_id'].unique(), warehouse_dates],
#         names=['unique_id', 'date']
#     ).to_frame(index=False)

#     # Add the warehouse column to the full combinations
#     full_combinations['warehouse'] = warehouse

#     # Merge with the warehouse sales data
#     merged_data = full_combinations.merge(
#         warehouse_sales_data,
#         on=['unique_id', 'warehouse', 'date'],
#         how='left'
#     )

#     # Fill missing sales with 0
#     merged_data['sales'].fillna(0, inplace=True)

#     # Collect the result
#     result_frames.append(merged_data)

#     # Trigger garbage collection to free up memory
#     #del full_combinations, warehouse_sales_data, merged_data
#     #gc.collect()

# # Step 3: Concatenate all results
# sales_complete = pd.concat(result_frames)
# sales_complete = sales_complete.reset_index(drop=True)

# # Display the sorted result
# print(sales_complete)



In [22]:
#sales_complete.to_csv('processed_sales_train_complete.csv', index=False)

In [26]:
sales_train=sales_train.drop('date', axis=1)

In [27]:
sales_train

,unique_id,warehouse,total_orders,sales,sell_price_main,product_unique_id,holiday_name,shops_closed,winter_school_holidays,school_holidays,year,month,day,day_of_week,day_before_holiday,day_after_holiday,max_discount
0,4845,Budapest_1,6436.0,16.34,646.26,2375,Not,0,0,0,2024,3,10,6,1.0,1.0,0.15312
1,4845,Budapest_1,4663.0,12.63,455.96,2375,Not,0,0,0,2021,5,25,1,1.0,1.0,0.15025
2,4845,Budapest_1,6507.0,34.55,455.96,2375,Not,0,0,0,2021,12,20,0,1.0,1.0,0.15025
3,4845,Budapest_1,5463.0,34.52,646.26,2375,Not,0,0,0,2023,4,29,5,1.0,1.0,0.20024
4,4845,Budapest_1,5997.0,35.92,486.41,2375,good_friday,0,0,0,2022,4,1,4,1.0,1.0,0.15649
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4007414,4941,Prague_1,9988.0,26.56,34.06,2422,Not,0,0,0,2023,6,21,2,1.0,1.0,0.00000
4007415,4941,Prague_1,8518.0,27.42,34.06,2422,Not,0,0,0,2023,6,24,5,1.0,1.0,0.00000
4007416,4941,Prague_1,10424.0,33.39,34.06,2422,Not,0,0,0,2023,6,23,4,1.0,1.0,0.00000
4007417,4941,Prague_1,10342.0,22.88,34.06,2422,Not,0,0,0,2023,6,22,3,1.0,1.0,0.00000


In [28]:
sales_train.columns

Index(['unique_id', 'warehouse', 'total_orders', 'sales', 'sell_price_main',
       'product_unique_id', 'holiday_name', 'shops_closed',
       'winter_school_holidays', 'school_holidays', 'year', 'month', 'day',
       'day_of_week', 'day_before_holiday', 'day_after_holiday',
       'max_discount'],
      dtype='object')

In [29]:
sales_train.to_csv('not_encoded_sales_train.csv', index=False)